# Лабораторная работа №13: Fine-tuning diffusion model

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Подготовить изображения/captions и fixed-seed baseline.
- Выполнить SD1.5 LoRA fine-tuning.
- Сравнить checkpoints, diversity и memorization.

## Теоретическая часть

Latent diffusion добавляет шум в latent space VAE, а U-Net предсказывает noise/velocity. LoRA адаптирует attention projections без изменения всех весов. Фиксированные seeds и prompts обязательны для сравнения checkpoints.

![Архитектура latent diffusion](https://raw.githubusercontent.com/CompVis/latent-diffusion/main/assets/modelfigure.png)


Latent diffusion обучает UNet предсказывать ε в пространстве VAE. LoRA добавляет обучаемые низкоранговые поправки к attention projections; VAE, text encoder и base UNet остаются замороженными. CLIP text similarity, разнообразие и ближайшая train генерация измеряют разные свойства и требуют ручного дополнения.

Для LoRA $W^{\prime}=W+BA$, где $B\in\mathbb{R}^{d\times r}$, $A\in\mathbb{R}^{r\times k}$, $r\ll\min(d,k)$. Noise-prediction минимизирует $\mathbb{E}_{x_0,\epsilon,t}\|\epsilon-\epsilon_\theta(z_t,t,c)\|_2^2$ на VAE-latents.

### Экспериментальный протокол

Основной эксперимент: SD 1.5 и 28 открытых реальных фотографий бабочек Wikimedia Commons при 512 px. `download_open_data.py` скачивает файлы и для каждого проверяет метаданные CC0-1.0 через Commons API, фиксирует страницу источника, URL лицензии и SHA256. Никакие файлы студента не требуются; `CV_LORA_DATA` задаёт альтернативный каталог с той же структурой metadata.jsonl. 3 независимых адаптера по 400 optimizer updates, checkpoint на 200/400; фиксированные held-out prompts и seeds. FAST_DEV_RUN: 256 px, один update, только smoke. Требуются GPU, Pillow, pip install diffusers transformers accelerate peft safetensors torchvision; на CPU не подменять обучение имитацией. CV_ARTIFACTS указывает на постоянный каталог Drive.

### Материалы

- [Diffusers LoRA training](https://huggingface.co/docs/diffusers/training/lora)
- [Latent Diffusion](https://arxiv.org/abs/2112.10752)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.

Версии библиотек, источник данных и лицензия должны быть указаны в отчёте.

## Реализация

`FAST_DEV_RUN=True` проверяет связность пайплайна на малом бюджете и не является итоговым экспериментом. Полная сдача требует `FAST_DEV_RUN=False` и реально полученных метрик.

In [1]:
from pathlib import Path
import os, json, random, time
os.environ.setdefault('HF_HUB_DISABLE_XET', '1')  # xet-загрузчик не нужен, классический CDN надёжнее за прокси
import numpy as np
import torch
SEED = 42
FAST_DEV_RUN = False  # True только для короткой проверки связности, не для сдачи.
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ARTIFACTS = Path(os.environ.get('CV_ARTIFACTS', 'artifacts'))
ARTIFACTS.mkdir(parents=True, exist_ok=True)
DATA_ROOT = Path(os.environ.get('CV_COURSE_DATA', str(Path.cwd().parents[1] / 'data')))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
def save_json(name, value):
    destination = ARTIFACTS / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")  # UTF-8 обязательно (Windows-локаль cp1251)
print({'device':DEVICE,'fast_dev_run':FAST_DEV_RUN,'artifacts':str(ARTIFACTS)})

{'device': 'cuda', 'fast_dev_run': False, 'artifacts': 'artifacts'}


In [2]:
# Сеть: если HTTPS перехватывается прокси/антивирусом (self-signed сертификат),
# переводим проверку сертификатов на системное хранилище ОС (пакет truststore,
# однократно: pip install truststore). Если пакета нет — стандартная проверка.
try:
    import truststore, urllib3
    truststore.inject_into_ssl()
    urllib3.disable_warnings()
    print("SSL: проверка сертификатов через системное хранилище ОС (truststore)")
except ImportError:
    print("SSL: truststore не установлен — стандартная проверка сертификатов")


SSL: проверка сертификатов через системное хранилище ОС (truststore)


In [3]:
# Проверка зависимостей лабы 13: diffusers, peft, safetensors, accelerate.
# При отсутствии: pip install -q "diffusers>=0.27" "peft>=0.11" safetensors accelerate
# (при проблемах с сертификатами добавьте --trusted-host pypi.org --trusted-host files.pythonhosted.org)
import importlib.util
_need = {"diffusers": '"diffusers>=0.27"', "peft": '"peft>=0.11"', "safetensors": "safetensors", "accelerate": "accelerate"}
missing = [k for k in _need if importlib.util.find_spec(k) is None]
if missing:
    raise ImportError("Установите недостающее и перезапустите ячейку:  pip install " + " ".join(_need[k] for k in missing))
import torch, transformers, diffusers, peft, safetensors
_info = {"torch": torch.__version__, "transformers": transformers.__version__,
         "diffusers": diffusers.__version__, "peft": peft.__version__}
if torch.cuda.is_available():
    _p = torch.cuda.get_device_properties(0)
    _free, _total = torch.cuda.mem_get_info()
    _info["gpu"] = f"{_p.name} | VRAM {_total/2**30:.1f} ГБ, свободно {_free/2**30:.1f} ГБ | cc {_p.major}.{_p.minor}"
    if _p.major < 7:
        _info["gpu_note"] = "Pascal: bf16 недоступен, используем fp16-основу + fp32-LoRA"
else:
    _info["gpu"] = "CUDA недоступна — обучение и генерация не пойдут; установите PyTorch с CUDA"
print("Версии:", _info)


W1028 14:53:35.257000 4704 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1028 14:53:35.337000 4704 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Версии: {'torch': '2.14.1+cu126', 'transformers': '4.57.6', 'diffusers': '0.39.0', 'peft': '0.21.2', 'gpu': 'NVIDIA GeForce GTX 1050 Ti | VRAM 4.0 ГБ, свободно 3.3 ГБ | cc 6.1', 'gpu_note': 'Pascal: bf16 недоступен, используем fp16-основу + fp32-LoRA'}


### Лицензия и валидация изображений

*Вес: 1 балл.*

**Постановка.** При малом наборе изображений утечка оценки в train captions делает сравнение LoRA некорректным. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Проверьте физические файлы, RGB-декодирование, уникальность и лицензии подписей.

**Результат.** Проверяемое состояние «Лицензия и валидация изображений» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Проверьте физические файлы, RGB-декодирование, уникальность и лицензии подписей» и его результат доступен следующему этапу.

In [4]:
# ЭТАП 1 — лицензия и валидация изображений: физические файлы, RGB-декод, уникальность, CC0.
# Логика скачивания — по materials/download_open_data.py (inline, без изменений семантики):
# лицензия проверяется по imageinfo.extmetadata для КАЖДОГО файла (не по результатам поиска);
# при изменении лицензии на стороне Commons загрузка падает — это требование протокола.
import hashlib, io
import urllib.request
from urllib.parse import urlencode
from PIL import Image

DATA_DIR = Path(os.environ.get('CV_LORA_DATA', 'data/lab13'))
_WMAPI = 'https://commons.wikimedia.org/w/api.php'
_HEADERS = {'User-Agent': 'CVUniversityLab/0.1 (educational research; contact: cv-lab@localhost.invalid)'}
_TITLES = [
    '20230130-butterfly.jpg', 'A Beautiful Black Butterfly.jpg',
    'A butterfly eating papaya, Location- Lakhimpur, Assam.jpg',
    'A butterfly on a purple flower.jpg', 'Black and white butterfly near Shirahama.jpg',
    'Butterfly 51.jpg', 'Butterfly Iguazu Falls.jpg', 'Butterfly The Insect.jpg',
    'Butterfly at Jonanjima.jpg', 'Butterfly at Wingham Wildlife Park Kent UK.jpg',
    'Butterfly by the water.jpg', 'Butterfly enjoying with nature.jpg',
    'Butterfly from butterfly garden-lucknow zoo.jpg', 'Butterfly hanging on leaf.jpg',
    'Butterfly in Armenia.jpg', 'Butterfly orange.jpg',
    'Common Jezebel Delias eucharis by kadavoor 3.jpg',
    'Common Pierrot butterfly at Anamangad, Malappuram, Kerala.jpg',
    'Common cerulean butterfly at Sooranad North.jpg',
    'Dark Cerulean Butterfly Female.jpg', 'Elymnias hypermnestra male by kadavoor.jpg',
    'Marbled White Butterfly (Melanargia galathea).jpg',
    'Monarch butterfly on butterfly bush flower.jpg',
    'Red Pierrot Butterfly at Jakkur, Bangalore.jpg',
    'Red admiral butterfly with wings closed.jpg', 'Swallowtail butterfly up close.jpg',
    'White butterfly upside down on leaf.jpg', 'Yellow and black butterfly on ground.jpg',
]

def _wm_fetch(url):
    with urllib.request.urlopen(urllib.request.Request(url, headers=_HEADERS), timeout=45) as resp:
        return resp.read()

def download_cc0_butterflies(destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    params = {'action': 'query', 'titles': '|'.join('File:' + t for t in _TITLES),
              'prop': 'imageinfo', 'iiprop': 'url|extmetadata|size|sha1|mime',
              'iiurlwidth': '640', 'format': 'json', 'formatversion': '2'}
    pages = {p['title']: p for p in json.loads(_wm_fetch(_WMAPI + '?' + urlencode(params)))['query']['pages']}
    if set(pages) != {'File:' + t for t in _TITLES}:
        raise ValueError('Commons изменил названия файлов: проверьте подборку вручную')
    records = []
    for index, title in enumerate(_TITLES):
        page = pages['File:' + title]
        info = page['imageinfo'][0]
        meta = info['extmetadata']
        lic_name, lic_url = meta['LicenseShortName']['value'], meta['LicenseUrl']['value']
        if lic_name != 'CC0' or 'creativecommons.org/publicdomain/zero/1.0/' not in lic_url:
            raise ValueError(f'Не подтверждена CC0 для {title}: {lic_name} {lic_url}')
        if info['mime'] != 'image/jpeg':
            raise ValueError(f'Не JPEG-фотография: {title}')
        filename = f'commons_{index:02d}.jpg'
        path = destination / filename
        if path.exists():
            raw = path.read_bytes()
        else:
            raw = _wm_fetch(info['thumburl'])
            with Image.open(io.BytesIO(raw)) as image:
                image.verify()
            path.write_bytes(raw)
        with Image.open(io.BytesIO(raw)) as image:
            image.convert('RGB').load()
            if min(image.size) < 256:
                raise ValueError(f'Слишком маленькое изображение: {title}: {image.size}')
        records.append({'file_name': filename,
                        'text': 'a photo of sks butterfly, ' + title.removesuffix('.jpg').replace('_', ' ').lower(),
                        'license': 'CC0-1.0', 'license_url': lic_url,
                        'source_url': info['descriptionurl'], 'download_url': info['thumburl'],
                        'original_sha1': info['sha1'], 'sha256': hashlib.sha256(raw).hexdigest(),
                        'source_title': page['title'],
                        'author_as_reported': meta.get('Artist', {}).get('value', '')})
        print(f'{index + 1}/{len(_TITLES)} {filename}: {title} ({len(raw)} bytes)')
    if len({r['sha256'] for r in records}) != len(records):
        raise ValueError('Повторяющиеся изображения')
    out = destination / 'metadata.jsonl'
    out.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in records), encoding='utf-8')
    print(f'{len(records)} реальных RGB JPEG, индивидуально подтверждённых Commons CC0; {out}')
    return records

t0 = time.time()
rows = download_cc0_butterflies(DATA_DIR)

# независимая валидация состояния на диске (критерий этапа)
sizes, sha_ok, lic_ok, rgb_ok = [], 0, 0, 0
for r in rows:
    path = DATA_DIR / r['file_name']
    assert path.exists(), f'нет файла {path}'
    with Image.open(path) as im:
        im.convert('RGB').load()
        rgb_ok += 1 if im.mode == 'RGB' else 0
        sizes.append(im.size)
    import hashlib as _h
    sha_ok += int(_h.sha256(path.read_bytes()).hexdigest() == r['sha256'])
    lic_ok += int(r['license'] == 'CC0-1.0' and 'creativecommons.org/publicdomain/zero/1.0/' in r['license_url'])
assert len(rows) == 28 and len({r['sha256'] for r in rows}) == 28 and len({r['text'] for r in rows}) == 28
assert sha_ok == 28 and lic_ok == 28 and rgb_ok == 28, (sha_ok, lic_ok, rgb_ok)
report = {'n_images': len(rows), 'sha256_verified': sha_ok, 'licenses_cc0': lic_ok, 'rgb': rgb_ok,
          'unique_sha256': len({r['sha256'] for r in rows}), 'unique_captions': len({r['text'] for r in rows}),
          'size_min': [min(w for w, h in sizes), min(h for w, h in sizes)],
          'size_max': [max(w for w, h in sizes), max(h for w, h in sizes)],
          'elapsed_s': round(time.time() - t0, 1), 'data_dir': str(DATA_DIR),
          'source': 'Wikimedia Commons, CC0-1.0, проверка по extmetadata каждого файла'}
save_json('lab13_data_report.json', report)
print(f"ЭТАП 1 OK: {len(rows)} фото (CC0 подтверждены по метаданным Commons, sha256 {sha_ok}/28, дубликатов 0); "
      f"размеры {report['size_min']}..{report['size_max']}; captions в metadata.jsonl; {report['elapsed_s']} c")


1/28 commons_00.jpg: 20230130-butterfly.jpg (135801 bytes)
2/28 commons_01.jpg: A Beautiful Black Butterfly.jpg (72201 bytes)
3/28 commons_02.jpg: A butterfly eating papaya, Location- Lakhimpur, Assam.jpg (124914 bytes)
4/28 commons_03.jpg: A butterfly on a purple flower.jpg (106346 bytes)
5/28 commons_04.jpg: Black and white butterfly near Shirahama.jpg (262645 bytes)
6/28 commons_05.jpg: Butterfly 51.jpg (90759 bytes)
7/28 commons_06.jpg: Butterfly Iguazu Falls.jpg (297914 bytes)
8/28 commons_07.jpg: Butterfly The Insect.jpg (354628 bytes)
9/28 commons_08.jpg: Butterfly at Jonanjima.jpg (93098 bytes)
10/28 commons_09.jpg: Butterfly at Wingham Wildlife Park Kent UK.jpg (84272 bytes)
11/28 commons_10.jpg: Butterfly by the water.jpg (152912 bytes)
12/28 commons_11.jpg: Butterfly enjoying with nature.jpg (101671 bytes)
13/28 commons_12.jpg: Butterfly from butterfly garden-lucknow zoo.jpg (497712 bytes)
14/28 commons_13.jpg: Butterfly hanging on leaf.jpg (163975 bytes)
15/28 commons_14.jp

### Отложенные текстовые запросы

*Вес: 1 балл.*

**Постановка.** При малом наборе изображений утечка оценки в train captions делает сравнение LoRA некорректным. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Зафиксируйте prompts и seeds, исключите точное совпадение с обучающими подписями.

**Результат.** Проверяемое состояние «Отложенные текстовые запросы» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Зафиксируйте prompts и seeds, исключите точное совпадение с обучающими подписями» и его результат доступен следующему этапу.

In [5]:
# ЭТАП 2 — отложенные текстовые запросы: prompts и seeds фиксированы ДО обучения;
# точные совпадения с обучающими подписями исключены (проверка ниже). Промпты намеренно
# покрывают канонический запрос, сцену, атрибут, счёт и стиль — часть заведомо «трудная».
EVAL_PROMPTS = [
    'a photo of sks butterfly',                                    # канонический (как train без суффикса)
    'a photo of sks butterfly on a flower',                        # сцена
    'a close-up photo of sks butterfly with open wings',           # атрибут
    'a photo of two sks butterflies sitting on a leaf',            # счёт (ожидаемо трудный)
    'a photo of sks butterfly in a summer garden',                 # сцена
    'a watercolor painting of sks butterfly on white background',  # стиль (ожидаемо трудный)
]
EVAL_SEEDS = [1234, 5678]
GEN_STEPS = int(os.environ.get('CV_SD_STEPS', '10' if FAST_DEV_RUN else '30'))
GUIDANCE = 7.5
RES = int(os.environ.get('CV_LORA_RES', '256' if FAST_DEV_RUN else '512'))
MODE = 'fast' if FAST_DEV_RUN else 'full'

train_texts = {r['text'] for r in rows}
assert not (set(EVAL_PROMPTS) & train_texts), 'утечка: eval-промпт совпадает с train-подписью'

_train_words = set(w for t in train_texts for w in t.lower().split())
novel_words = {p: sorted(set(p.lower().split()) - _train_words) for p in EVAL_PROMPTS}
protocol = {'mode': MODE, 'resolution': RES, 'steps': GEN_STEPS, 'guidance': GUIDANCE,
            'negative_prompt': '', 'prompts': EVAL_PROMPTS, 'seeds': EVAL_SEEDS,
            'n_train_captions': len(train_texts),
            'novel_words_vs_train': novel_words,
            'leak_check': 'exact-match excluded; для прозрачности — слова промптов, которых нет ни в одной train-подписи (новые условия)'}
save_json('lab13_eval_prompts.json', protocol)
_novel_all = sorted(set(w for v in novel_words.values() for w in v))
print(f'ЭТАП 2 OK: {len(EVAL_PROMPTS)} промптов x {len(EVAL_SEEDS)} seeds; точных совпадений с '
      f'{len(train_texts)} train-подписями: 0; новые условия, отсутствующие в train: {_novel_all}; '
      f'режим {MODE}: {RES}px, {GEN_STEPS} шагов, guidance {GUIDANCE}')


ЭТАП 2 OK: 6 промптов x 2 seeds; точных совпадений с 28 train-подписями: 0; новые условия, отсутствующие в train: ['background', 'butterflies', 'close-up', 'garden', 'open', 'painting', 'sitting', 'summer', 'two', 'watercolor']; режим full: 512px, 30 шагов, guidance 7.5


### Baseline SD на фиксированных запросах

*Вес: 1 балл.*

**Постановка.** Baseline до обучения и сопоставимый inference protocol позволяют увидеть фактический вклад LoRA. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Загрузите исходную SD1.5 на GPU и сгенерируйте baseline до установки адаптера на фиксированных запросах/seeds.

**Результат.** Проверяемое состояние «Baseline SD на фиксированных запросах» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Загрузите исходную SD1.5 на GPU и сгенерируйте baseline до установки адаптера на фиксированных запросах/seeds» и его результат доступен следующему этапу.

In [6]:
# ЭТАП 3 — baseline SD1.5 (БЕЗ LoRA) на фиксированных запросах/seed + кэш text-эмбеддингов.
# Железо 4 ГБ: пайплайн держим на CPU; на GPU поднимаем только нужный компонент.
# Генерация — ручной цикл DPM++ (детерминированный seed), text encoder в инференсе не участвует.
import gc
from diffusers import DPMSolverMultistepScheduler, StableDiffusionPipeline
from torchvision.transforms.functional import to_pil_image

SD_ID = os.getenv('CV_SD_MODEL', 'stable-diffusion-v1-5/stable-diffusion-v1-5')
assert torch.cuda.is_available(), 'Нужен GPU (см. постановку): на CPU обучение не подменяется имитацией.'
DEV = 'cuda'

try:
    SD_PIPE = StableDiffusionPipeline.from_pretrained(SD_ID, torch_dtype=torch.float16,
                                                      safety_checker=None, requires_safety_checker=False,
                                                      variant='fp16')
    print('SD1.5: загружены fp16-веса (variant=fp16)')
except Exception as e:
    print(f'variant=fp16 недоступен ({type(e).__name__}) — загружаем основные веса')
    SD_PIPE = StableDiffusionPipeline.from_pretrained(SD_ID, torch_dtype=torch.float16,
                                                      safety_checker=None, requires_safety_checker=False)
SD_PIPE.set_progress_bar_config(disable=True)
SD_PIPE.enable_attention_slicing()
SD_PIPE.enable_vae_slicing()

# --- кэш text-эмбеддингов: 28 подписей + eval-промпты + пустой (unconditional) ---
SD_PIPE.text_encoder.to(DEV)
def encode_texts(texts):
    ids = SD_PIPE.tokenizer(list(texts), padding='max_length', truncation=True,
                            max_length=77, return_tensors='pt').input_ids.to(DEV)
    with torch.no_grad():
        return SD_PIPE.text_encoder(ids)[0].detach().cpu()
_train_texts = [r['text'] for r in rows]
_all_texts = _train_texts + list(EVAL_PROMPTS) + ['']
_emb = encode_texts(_all_texts)
TEXT_EMBEDS = {t: _emb[i] for i, t in enumerate(_all_texts)}
UNCOND_EMBED = TEXT_EMBEDS['']
SD_PIPE.text_encoder.to('cpu'); gc.collect(); torch.cuda.empty_cache()
print(f'кэш text-эмбеддингов: {len(TEXT_EMBEDS)} строк x 77 x 768 (fp16, CPU)')

# --- ручной цикл генерации: UNet+VAE на GPU, scheduler на CPU, эмбеддинги готовые ---
GEN_SCHED = DPMSolverMultistepScheduler.from_config(SD_PIPE.scheduler.config)

def sd_generate(unet, vae, cond_embed, seed, height=None, width=None, steps=None, guidance=None):
    """Одно изображение 512px (или RES) при фиксированном seed; CFG вручную."""
    height = height or RES; width = width or RES
    steps = steps or GEN_STEPS; guidance = GUIDANCE if guidance is None else guidance
    gen = torch.Generator(device=DEV); gen.manual_seed(int(seed))
    lat = torch.randn((1, 4, height // 8, width // 8), generator=gen, device=DEV, dtype=torch.float32)
    emb = torch.cat([UNCOND_EMBED[None], cond_embed[None]], 0).half().to(DEV)  # [2, 77, 768]
    GEN_SCHED.set_timesteps(steps)
    with torch.no_grad():
        for t in GEN_SCHED.timesteps:
            zin = lat.half().repeat(2, 1, 1, 1)
            noise_pred = unet(zin, t.expand(2).to(DEV), emb).sample.float()
            nu, nc = noise_pred.chunk(2)
            noise_pred = nu + guidance * (nc - nu)
            lat = GEN_SCHED.step(noise_pred, t, lat).prev_sample
        img = vae.decode(lat.half() / vae.config.scaling_factor).sample
    img = (img.float() / 2 + 0.5).clamp(0, 1)
    return to_pil_image(img[0].cpu())

# --- baseline ДО установки адаптера (тот же набор prompts/seeds, что и у адаптеров) ---
SD_PIPE.unet.to(DEV); SD_PIPE.vae.to(DEV)
LAB13 = ARTIFACTS / 'lab13'; (LAB13 / 'gen').mkdir(parents=True, exist_ok=True)
BASE_DIR = LAB13 / 'gen' / f'{MODE}_baseline'; BASE_DIR.mkdir(parents=True, exist_ok=True)
_pl = list(EVAL_PROMPTS[:1]) if FAST_DEV_RUN else list(EVAL_PROMPTS)
_sl = list(EVAL_SEEDS[:1]) if FAST_DEV_RUN else list(EVAL_SEEDS)
baseline_manifest, t0 = [], time.time()
for pi, prompt in enumerate(_pl):
    for seed in _sl:
        path = BASE_DIR / f'p{pi:02d}_seed{seed}.png'
        ts = time.time()
        if not path.exists():
            sd_generate(SD_PIPE.unet, SD_PIPE.vae, TEXT_EMBEDS[prompt], seed).save(path)
        baseline_manifest.append({'checkpoint': 'baseline', 'step': None, 'prompt': prompt,
                                  'prompt_idx': pi, 'seed': seed, 'path': str(path),
                                  'seconds': round(time.time() - ts, 1)})
        print(f'[baseline] {pi + 1}/{len(_pl)} seed {seed} | {time.time() - ts:.0f} c | всего {time.time() - t0:.0f} c')
(LAB13 / 'gen_manifest_baseline.jsonl').write_text(
    ''.join(json.dumps(m, ensure_ascii=False) + '\n' for m in baseline_manifest), encoding='utf-8')
SD_PIPE.unet.to('cpu'); SD_PIPE.vae.to('cpu'); gc.collect(); torch.cuda.empty_cache()
_per_img = baseline_manifest[0]['seconds']
_est = len(_pl) * len(_sl) * _per_img
print(f'ЭТАП 3 OK: baseline {len(baseline_manifest)} изображений (SD1.5 fp16, {RES}px, {GEN_STEPS} шагов DPM++, '
      f'guidance {GUIDANCE}, без LoRA) -> {BASE_DIR}; ~{_per_img:.0f} c/изображение; '
      f'baseline регенерации не требует при перезапуске (файлы кэшируются)')


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

SD1.5: загружены fp16-веса (variant=fp16)


c:\Users\eric1\.conda\envs\fmi-cv\Lib\site-packages\diffusers\pipelines\pipeline_utils.py:2273: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


кэш text-эмбеддингов: 35 строк x 77 x 768 (fp16, CPU)
[baseline] 1/6 seed 1234 | 48 c | всего 48 c
[baseline] 1/6 seed 5678 | 48 c | всего 96 c
[baseline] 2/6 seed 1234 | 47 c | всего 144 c
[baseline] 2/6 seed 5678 | 47 c | всего 191 c
[baseline] 3/6 seed 1234 | 47 c | всего 238 c
[baseline] 3/6 seed 5678 | 47 c | всего 285 c
[baseline] 4/6 seed 1234 | 47 c | всего 332 c
[baseline] 4/6 seed 5678 | 47 c | всего 379 c
[baseline] 5/6 seed 1234 | 48 c | всего 428 c
[baseline] 5/6 seed 5678 | 48 c | всего 476 c
[baseline] 6/6 seed 1234 | 48 c | всего 524 c
[baseline] 6/6 seed 5678 | 49 c | всего 573 c
ЭТАП 3 OK: baseline 12 изображений (SD1.5 fp16, 512px, 30 шагов DPM++, guidance 7.5, без LoRA) -> artifacts\lab13\gen\full_baseline; ~48 c/изображение; baseline регенерации не требует при перезапуске (файлы кэшируются)


### Подготовка пар изображение–caption

*Вес: 1 балл.*

**Постановка.** Различия rank и learning rate оцениваются при одинаковых изображениях, количестве optimizer updates и eval prompts. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Соберите Dataset с препроцессингом и токенизацией; проверьте длину и диапазон.

**Результат.** Проверяемое состояние «Подготовка пар изображение–caption» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Соберите Dataset с препроцессингом и токенизацией; проверьте длину и диапазон» и его результат доступен следующему этапу.

In [7]:
# ЭТАП 4 — Dataset пар изображение-caption: препроцессинг, токенизация, проверки длины/диапазона;
# попутно детерминированный кэш VAE-latents (одинаковые для всех конфигураций LoRA).
from torch.utils.data import Dataset
from torchvision import transforms

class ButterflyPairs(Dataset):
    """Пар изображение-caption: пиксели [-1, 1] (SD-конвенция) + CLIP-токены (77)."""
    def __init__(self, rows, resolution, tokenizer):
        self.rows = rows; self.res = resolution; self.tok = tokenizer
        self.tf = transforms.Compose([
            transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.CenterCrop(resolution), transforms.ToTensor()])
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        with Image.open(DATA_DIR / r['file_name']) as im:
            px = self.tf(im.convert('RGB'))
        ids = self.tok(r['text'], padding='max_length', truncation=True,
                       max_length=77, return_tensors='pt').input_ids[0]
        return {'pixel_values': px * 2 - 1, 'input_ids': ids, 'id': r['file_name']}

ds = ButterflyPairs(rows, RES, SD_PIPE.tokenizer)
assert len(ds) == 28

# проверки длины токенов и диапазона пикселей (критерий этапа)
_tok_lens = [int((ds[i]['input_ids'] != SD_PIPE.tokenizer.pad_token_id).sum()) for i in range(len(ds))]
_px = torch.stack([ds[i]['pixel_values'] for i in range(len(ds))])
assert max(_tok_lens) <= 77 and min(_tok_lens) >= 2, _tok_lens
assert float(_px.min()) >= -1.0 and float(_px.max()) <= 1.0

# --- кэш VAE-latents (fp16, CPU): детерминированно, .mode() без сэмплирования ---
lat_path = ARTIFACTS / f'lab13_latents_{RES}.npz'
_cached = lat_path.exists()
if _cached:
    _z = np.load(lat_path)
    LATENTS, LAT_IDS = _z['X'], list(str(x) for x in _z['ids'])
    assert LATENTS.shape == (28, 4, RES // 8, RES // 8) and LAT_IDS == [r['file_name'] for r in rows]
else:
    SD_PIPE.vae.to(DEV)
    LATENTS = np.zeros((len(rows), 4, RES // 8, RES // 8), dtype=np.float16)
    with torch.no_grad():
        for i in range(len(ds)):
            px = ds[i]['pixel_values'].unsqueeze(0).to(DEV, torch.float16)
            lat = SD_PIPE.vae.encode(px).latent_dist.mode() * SD_PIPE.vae.config.scaling_factor
            LATENTS[i] = lat.squeeze(0).float().cpu().numpy().astype(np.float16)
    SD_PIPE.vae.to('cpu'); gc.collect(); torch.cuda.empty_cache()
    np.savez_compressed(lat_path, X=LATENTS, ids=np.array([r['file_name'] for r in rows]))
    LAT_IDS = [r['file_name'] for r in rows]

report = {'n_pairs': len(ds), 'resolution': RES, 'token_len_min': int(min(_tok_lens)),
          'token_len_mean': round(float(np.mean(_tok_lens)), 1), 'token_len_max': int(max(_tok_lens)),
          'pixel_min': round(float(_px.min()), 4), 'pixel_max': round(float(_px.max()), 4),
          'latents_shape': list(LATENTS.shape), 'latents_from_cache': _cached}
save_json('lab13_dataset_report.json', report)
print(f"ЭТАП 4 OK: Dataset 28 пар ({RES}px); токены {_tok_lens and (min(_tok_lens), max(_tok_lens))} <= 77; "
      f"пиксели [{report['pixel_min']}, {report['pixel_max']}] в [-1, 1]; "
      f"latents {LATENTS.shape} -> {lat_path.name} {'(из кэша)' if _cached else ''}")


ЭТАП 4 OK: Dataset 28 пар (512px); токены (9, 22) <= 77; пиксели [-1.0, 1.0] в [-1, 1]; latents (28, 4, 64, 64) -> lab13_latents_512.npz 


### Заморозка основы и конфигурации LoRA

*Вес: 1 балл.*

**Постановка.** Различия rank и learning rate оцениваются при одинаковых изображениях, количестве optimizer updates и eval prompts. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Заморозьте основу, установите тренировочный schedule, выберите три конфигурации rank/LR.

**Результат.** Проверяемое состояние «Заморозка основы и конфигурации LoRA» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Заморозьте основу, установите тренировочный schedule, выберите три конфигурации rank/LR» и его результат доступен следующему этапу.

In [8]:
# ЭТАП 5 — заморозка основы, тренировочный schedule, ТРИ конфигурации rank/LR.
# Различия конфигураций оцениваются на одинаковых изображениях, бюджете updates и eval-промптах.
for _m in ('unet', 'text_encoder', 'vae'):
    for _p in getattr(SD_PIPE, _m).parameters():
        _p.requires_grad_(False)
_frozen = all(not p.requires_grad for m in ('unet', 'text_encoder', 'vae') for p in getattr(SD_PIPE, m).parameters())
assert _frozen, 'основа должна быть полностью заморожена'

N_UPDATES = 1 if FAST_DEV_RUN else 400          # optimizer updates на адаптер
CKPT_STEPS = (1,) if FAST_DEV_RUN else (200, 400)
TRAIN_TARGETS = ['to_q', 'to_k', 'to_v', 'to_out.0']   # attention projections UNet
LORA_CONFIGS = [
    {'name': 'r8_lr1e-4',  'rank': 8,  'lr': 1e-4, 'alpha': 8},
    {'name': 'r16_lr1e-4', 'rank': 16, 'lr': 1e-4, 'alpha': 16},
    {'name': 'r8_lr5e-4',  'rank': 8,  'lr': 5e-4, 'alpha': 8},
] if not FAST_DEV_RUN else [{'name': 'fast_smoke', 'rank': 8, 'lr': 1e-4, 'alpha': 8}]

PROTOCOL = {'mode': MODE, 'seed': SEED, 'resolution': RES, 'batch': 1, 'n_updates': N_UPDATES,
            'ckpt_steps': list(CKPT_STEPS), 'lora_targets': TRAIN_TARGETS, 'lora_dropout': 0.0,
            'optimizer': 'AdamW', 'betas': [0.9, 0.999], 'weight_decay': 0.01, 'grad_clip': 1.0,
            'lr_schedule': 'constant', 'precision': 'fp16-основа + fp32-LoRA (без GradScaler)',
            'gradient_checkpointing': True, 'attention_slicing': True,
            'uncond_dropout_train': 0.0, 'sd_model': SD_ID, 'clip_model': 'openai/clip-vit-base-patch32',
            'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}
save_json('lab13_lora_configs.json', {'protocol': PROTOCOL, 'configs': LORA_CONFIGS})
print(f'ЭТАП 5 OK: основа заморожена (unet/text_encoder/vae); targets {TRAIN_TARGETS}; '
      f'{len(LORA_CONFIGS)} конфигурации: ' + '; '.join(f"{c['name']} (r={c['rank']}, lr={c['lr']})" for c in LORA_CONFIGS) +
      f'; AdamW, {N_UPDATES} updates, checkpoint на {list(CKPT_STEPS)}')


ЭТАП 5 OK: основа заморожена (unet/text_encoder/vae); targets ['to_q', 'to_k', 'to_v', 'to_out.0']; 3 конфигурации: r8_lr1e-4 (r=8, lr=0.0001); r16_lr1e-4 (r=16, lr=0.0001); r8_lr5e-4 (r=8, lr=0.0005); AdamW, 400 updates, checkpoint на [200, 400]


### Оптимизация адаптеров и checkpoint

*Вес: 1 балл.*

**Постановка.** Различия rank и learning rate оцениваются при одинаковых изображениях, количестве optimizer updates и eval prompts. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Обучите независимые адаптеры, сохраняйте optimizer, scaler и переносимые веса при каждом контрольном шаге.

**Результат.** Проверяемое состояние «Оптимизация адаптеров и checkpoint» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Обучите независимые адаптеры, сохраняйте optimizer, scaler и переносимые веса при каждом контрольном шаге» и его результат доступен следующему этапу.

In [9]:
# ЭТАП 6 — обучение независимых адаптеров: AdamW, fp16-основа + fp32-LoRA, checkpoint 200/400
# с optimizer+rng и resume; шаги полностью детерминированы (seed зависит от номера шага).
import safetensors.torch
from diffusers import DDPMScheduler
from peft import (LoraConfig, get_peft_model, get_peft_model_state_dict,
                  set_peft_model_state_dict)

CKPT_ROOT = ARTIFACTS / 'lab13' / 'ckpt'
UNET_PEFT = None
noise_scheduler = DDPMScheduler.from_config(SD_PIPE.scheduler.config)
SD_PIPE.unet.to(DEV)
SD_PIPE.unet.enable_gradient_checkpointing()

def _lora_cfg(c):
    return LoraConfig(r=c['rank'], lora_alpha=c['alpha'], lora_dropout=0.0,
                      target_modules=TRAIN_TARGETS, init_lora_weights=True)

def _add_adapter_compat(pm, name, cfg):
    """порядок аргументов add_adapter отличается между версиями peft."""
    import inspect as _inspect
    _pars = list(_inspect.signature(type(pm).add_adapter).parameters)
    if _pars[1:3] == ['adapter_name', 'peft_config']:      # новые peft (0.16+)
        pm.add_adapter(name, cfg)
    else:                                                  # старые peft: (adapter_config, adapter_name)
        pm.add_adapter(cfg, adapter_name=name)

def ensure_adapter(c):
    """Один активный адаптер за раз; добавление нового не портит существующие."""
    global UNET_PEFT
    if UNET_PEFT is None:
        UNET_PEFT = get_peft_model(SD_PIPE.unet, _lora_cfg(c), adapter_name=c['name'])
    elif c['name'] not in UNET_PEFT.peft_config:
        _add_adapter_compat(UNET_PEFT, c['name'], _lora_cfg(c))
    UNET_PEFT.set_adapter(c['name'])
    return UNET_PEFT

def save_adapter_weights(c, out_dir, extra):
    out_dir.mkdir(parents=True, exist_ok=True)
    sd = get_peft_model_state_dict(UNET_PEFT, adapter_name=c['name'])
    safetensors.torch.save_file(sd, out_dir / 'adapter_model.safetensors')
    (out_dir / 'adapter_config.json').write_text(
        json.dumps({'adapter_name': c['name'], **c, 'targets': TRAIN_TARGETS, **extra},
                   ensure_ascii=False, indent=2), encoding='utf-8')

def load_adapter_weights(c, in_dir):
    ensure_adapter(c)
    sd = safetensors.torch.load_file(Path(in_dir) / 'adapter_model.safetensors')
    res = set_peft_model_state_dict(UNET_PEFT, sd, adapter_name=c['name'])
    # missing_keys peft всегда перечисляет замороженные base_layer веса, а при нескольких
    # адаптерах — ещё и lora-ключи ДРУГИХ адаптеров; ошибкой является отсутствие
    # lora-ключей именно текущего адаптера или неожиданные ключи
    _miss = [k for k in getattr(res, 'missing_keys', [])
             if 'lora' in k and f".{c['name']}." in k]
    _unexp = list(getattr(res, 'unexpected_keys', []))
    assert not _miss and not _unexp, f'адаптер {c["name"]}: missing {_miss[:3]} unexpected {_unexp[:3]}'
    UNET_PEFT.set_adapter(c['name'])

def _resume_state(name, ckpt_steps):
    best = None
    for s in ckpt_steps:
        d = CKPT_ROOT / name / f'step{s}'
        if (d / 'adapter_model.safetensors').exists() and (d / 'optimizer_state.pt').exists():
            best = (s, d)
    return best

def train_adapter(c, ckpt_steps=None, p_uncond=0.0, seed_base=None):
    """400 updates на кэшированных latents/эмбеддингах; checkpoint каждые ckpt_steps; resume."""
    ckpt_steps = tuple(CKPT_STEPS) if ckpt_steps is None else tuple(ckpt_steps)
    seed_base = SEED if seed_base is None else seed_base
    unet_peft = ensure_adapter(c)
    for _n, _p in unet_peft.named_parameters():
        _p.requires_grad_('lora' in _n and f".{c['name']}." in _n)
    params = [_p for _p in unet_peft.parameters() if _p.requires_grad]
    assert params, 'нет обучаемых параметров LoRA'
    print(f"[{c['name']}] обучаемых параметров: {sum(p.numel() for p in params) / 1e6:.2f}M "
          f"(rank {c['rank']}, lr {c['lr']})")
    _free, _total = torch.cuda.mem_get_info()
    print(f"[{c['name']}] VRAM свободно {_free/2**30:.1f} ГБ из {_total/2**30:.1f} ГБ")
    opt = torch.optim.AdamW(params, lr=c['lr'], betas=(0.9, 0.999), weight_decay=0.01)
    start, losses = 0, []
    rs = _resume_state(c['name'], ckpt_steps)
    if rs:
        load_adapter_weights(c, rs[1])
        st = torch.load(rs[1] / 'optimizer_state.pt', map_location='cpu', weights_only=False)
        opt.load_state_dict(st['optimizer']); start = st['step']
        torch.set_rng_state(st['rng_torch'].cpu()); random.setstate(st['rng_py']); np.random.set_state(st['rng_np'])
        if DEV == 'cuda' and st.get('rng_cuda') is not None:
            torch.cuda.set_rng_state(st['rng_cuda'].cpu())
        print(f"[{c['name']}] возобновление с шага {start} ({rs[1]})")
    if start < N_UPDATES:
        unet_peft.train()
        t0 = time.time()
        for s in range(start, N_UPDATES):
            epoch = s // len(ds)
            order = list(range(len(ds))); random.Random(SEED * 1009 + epoch).shuffle(order)
            idx = order[s % len(ds)]
            lat = torch.tensor(LATENTS[idx]).unsqueeze(0).to(DEV, torch.float16)
            g = torch.Generator(device=DEV); g.manual_seed(seed_base + s * 9_977)
            t = torch.randint(0, 1000, (1,), generator=g, device=DEV)
            noise = torch.randn(lat.shape, generator=g, device=DEV, dtype=torch.float16)
            cond_key = rows[idx]['text']
            if p_uncond > 0 and float(torch.rand((), generator=g, device=DEV)) < p_uncond:
                cond_key = ''                       # CFG-dropout: шаг без условия
            emb = TEXT_EMBEDS[cond_key].unsqueeze(0).half().to(DEV)
            noisy = noise_scheduler.add_noise(lat, noise, t).half()
            pred = unet_peft(noisy, t, emb).sample
            loss = torch.nn.functional.mse_loss(pred.float(), noise.float())
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            if (s + 1) % 25 == 0 or (s + 1) == N_UPDATES:
                losses.append({'step': s + 1, 'loss': round(float(loss.item()), 4)})
            if (s + 1) in ckpt_steps or (s + 1) == N_UPDATES:
                d = CKPT_ROOT / c['name'] / f'step{s + 1}'
                save_adapter_weights(c, d, {'step': s + 1, 'p_uncond': p_uncond,
                                            'elapsed_s': round(time.time() - t0, 1)})
                torch.save({'optimizer': opt.state_dict(), 'step': s + 1,
                            'rng_torch': torch.get_rng_state().cpu(),
                            'rng_cuda': torch.cuda.get_rng_state().cpu() if DEV == 'cuda' else None,
                            'rng_py': random.getstate(), 'rng_np': np.random.get_state()},
                           d / 'optimizer_state.pt')
            if s % 25 == 0:
                done = s + 1 - start; rate = (time.time() - t0) / max(done, 1)
                print(f"[{c['name']}] шаг {s + 1}/{N_UPDATES} | loss {float(loss.item()):.4f} | "
                      f"{rate:.2f} c/шаг | ETA {(N_UPDATES - s - 1) * rate / 60:.0f} мин")
        res = {'config': c, 'p_uncond': p_uncond, 'losses': losses,
               'elapsed_s': round(time.time() - t0, 1), 'resumed_from': start}
        (CKPT_ROOT / c['name'] / f'step{N_UPDATES}' / 'meta.json').write_text(
            json.dumps(res, ensure_ascii=False, indent=2), encoding='utf-8')
        return res
    _meta = CKPT_ROOT / c['name'] / f'step{N_UPDATES}' / 'meta.json'
    if _meta.exists():
        res = json.loads(_meta.read_text(encoding='utf-8'))
        res['already_done'] = True
        return res
    return {'config': c, 'p_uncond': p_uncond, 'losses': [], 'elapsed_s': 0,
            'resumed_from': start, 'already_done': True}

training_log = {}
_t6 = time.time()
for _ci, _c in enumerate(LORA_CONFIGS):
    training_log[_c['name']] = train_adapter(_c, seed_base=SEED + _ci * 1_000_003)
save_json('lab13_training_log.json', training_log)
print(f'ЭТАП 6 OK: обучено {len(LORA_CONFIGS)} адаптеров по {N_UPDATES} updates '
      f'({time.time() - _t6:.0f} c суммарно); финальные loss: ' +
      '; '.join(f"{k} {v['losses'][-1]['loss'] if v['losses'] else 'n/a'}" for k, v in training_log.items()) +
      f' -> artifacts/lab13_training_log.json; checkpoints -> {CKPT_ROOT}')


[r8_lr1e-4] обучаемых параметров: 1.59M (rank 8, lr 0.0001)
[r8_lr1e-4] VRAM свободно 1.6 ГБ из 4.0 ГБ
[r8_lr1e-4] шаг 1/400 | loss 0.0357 | 2.61 c/шаг | ETA 17 мин
[r8_lr1e-4] шаг 26/400 | loss 0.1307 | 2.33 c/шаг | ETA 15 мин
[r8_lr1e-4] шаг 51/400 | loss 0.0213 | 2.33 c/шаг | ETA 14 мин
[r8_lr1e-4] шаг 76/400 | loss 0.0455 | 2.33 c/шаг | ETA 13 мин
[r8_lr1e-4] шаг 101/400 | loss 0.4695 | 2.33 c/шаг | ETA 12 мин
[r8_lr1e-4] шаг 126/400 | loss 0.0033 | 2.33 c/шаг | ETA 11 мин
[r8_lr1e-4] шаг 151/400 | loss 0.0698 | 2.33 c/шаг | ETA 10 мин
[r8_lr1e-4] шаг 176/400 | loss 0.1410 | 2.33 c/шаг | ETA 9 мин
[r8_lr1e-4] шаг 201/400 | loss 0.2258 | 2.32 c/шаг | ETA 8 мин
[r8_lr1e-4] шаг 226/400 | loss 0.4190 | 2.32 c/шаг | ETA 7 мин
[r8_lr1e-4] шаг 251/400 | loss 0.4168 | 2.32 c/шаг | ETA 6 мин
[r8_lr1e-4] шаг 276/400 | loss 0.2565 | 2.32 c/шаг | ETA 5 мин
[r8_lr1e-4] шаг 301/400 | loss 0.1033 | 2.32 c/шаг | ETA 4 мин
[r8_lr1e-4] шаг 326/400 | loss 0.0292 | 2.32 c/шаг | ETA 3 мин
[r8_lr1e-4] ш

### Признаки CLIP для независимой оценки

*Вес: 1 балл.*

**Постановка.** Одна similarity-метрика не определяет качество генерации: высокое совпадение с текстом может сопровождаться копированием train. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Загрузите CLIP и определите нормализованные текстовые и визуальные embeddings.

**Результат.** Проверяемое состояние «Признаки CLIP для независимой оценки» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Загрузите CLIP и определите нормализованные текстовые и визуальные embeddings» и его результат доступен следующему этапу.

In [10]:
# ЭТАП 7 — CLIP ViT-B/32 для независимой оценки: нормализованные текстовые и визуальные эмбеддинги.
# CLIP держим на CPU: на карте 4 ГБ каждый гигабайт нужен SD; скорости CPU достаточно.
from transformers import CLIPModel, CLIPProcessor

CLIP_ID = os.getenv('CV_CLIP_MODEL', 'openai/clip-vit-base-patch32')
CLIP = CLIPModel.from_pretrained(CLIP_ID).eval()
CLIP_PROC = CLIPProcessor.from_pretrained(CLIP_ID)
assert next(CLIP.parameters()).device.type == 'cpu', 'CLIP должен оставаться на CPU'

@torch.no_grad()
def clip_image_embed(path):
    with Image.open(path) as im:
        pv = CLIP_PROC(images=im.convert('RGB'), return_tensors='pt')['pixel_values']
    e = CLIP.get_image_features(pv)
    return torch.nn.functional.normalize(e, dim=-1)[0].numpy().astype('float32')

@torch.no_grad()
def clip_text_embed(text):
    enc = CLIP_PROC(text=[text], padding=True, truncation=True, return_tensors='pt')
    e = CLIP.get_text_features(**{k: v for k, v in enc.items() if k in ('input_ids', 'attention_mask')})
    return torch.nn.functional.normalize(e, dim=-1)[0].numpy().astype('float32')

_a = clip_image_embed(DATA_DIR / rows[0]['file_name'])
_own = float(np.dot(_a, clip_text_embed(rows[0]['text'])))
_oth = float(np.dot(_a, clip_text_embed(rows[5]['text'])))
assert _own > _oth, (_own, _oth)
save_json('lab13_clip_check.json', {'model': CLIP_ID, 'device': 'cpu',
                                    'cos_own_caption': round(_own, 4), 'cos_other_caption': round(_oth, 4)})
print(f'ЭТАП 7 OK: CLIP ViT-B/32 (CPU), нормализованные text/image эмбеддинги; '
      f'smoke: cos(фото, своя подпись) {_own:.3f} > cos(чужая подпись) {_oth:.3f}')


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


ЭТАП 7 OK: CLIP ViT-B/32 (CPU), нормализованные text/image эмбеддинги; smoke: cos(фото, своя подпись) 0.300 > cos(чужая подпись) 0.251


### Поиск ближайших обучающих изображений

*Вес: 1 балл.*

**Постановка.** Одна similarity-метрика не определяет качество генерации: высокое совпадение с текстом может сопровождаться копированием train. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Вычислите эмбеддинги реальных train изображений и зафиксируйте набор checkpoint для оценки.

**Результат.** Проверяемое состояние «Поиск ближайших обучающих изображений» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Вычислите эмбеддинги реальных train изображений и зафиксируйте набор checkpoint для оценки» и его результат доступен следующему этапу.

In [11]:
# ЭТАП 8 — эмбеддинги реальных train изображений (для метрики memorization «ближайший train»)
# и реестр checkpoint, выносимых на оценку: baseline + каждая конфигурация x каждый контрольный шаг.
emb_path = ARTIFACTS / 'lab13_train_clip.npz'
if emb_path.exists():
    _z = np.load(emb_path)
    TRAIN_CLIP, TRAIN_CLIP_IDS = _z['X'], [str(x) for x in _z['ids']]
else:
    TRAIN_CLIP = np.stack([clip_image_embed(DATA_DIR / r['file_name']) for r in rows]).astype('float32')
    TRAIN_CLIP_IDS = [r['file_name'] for r in rows]
    np.savez(emb_path, X=TRAIN_CLIP, ids=np.array(TRAIN_CLIP_IDS))
assert TRAIN_CLIP.shape == (len(rows), 512), TRAIN_CLIP.shape

EVAL_REGISTRY = [{'checkpoint': 'baseline', 'step': None, 'adapter_dir': None, 'config': None}]
EVAL_REGISTRY += [{'checkpoint': c['name'], 'step': s,
                   'adapter_dir': str(CKPT_ROOT / c['name'] / f'step{s}'), 'config': c}
                  for c in LORA_CONFIGS for s in CKPT_STEPS]
for e in EVAL_REGISTRY:
    if e['adapter_dir']:
        e['ckpt_present'] = (Path(e['adapter_dir']) / 'adapter_model.safetensors').exists()
save_json('lab13_checkpoint_registry.json', EVAL_REGISTRY)
_all_present = all(e.get('ckpt_present', True) for e in EVAL_REGISTRY)
print(f'ЭТАП 8 OK: {len(rows)} train CLIP-эмбеддингов (512-d, CPU); реестр оценки: {len(EVAL_REGISTRY)} '
      f'checkpoint ({len(EVAL_REGISTRY) - 1} адаптерных); adapter-файлы на месте: {_all_present}')


ЭТАП 8 OK: 28 train CLIP-эмбеддингов (512-d, CPU); реестр оценки: 7 checkpoint (6 адаптерных); adapter-файлы на месте: True


### Парные генерации и измеримые показатели

*Вес: 1 балл.*

**Постановка.** Одна similarity-метрика не определяет качество генерации: высокое совпадение с текстом может сопровождаться копированием train. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Для равных prompts/seeds измерьте текстовое сходство, разнообразие и ближайший train объект; baseline не должен содержать LoRA.

**Результат.** Проверяемое состояние «Парные генерации и измеримые показатели» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Для равных prompts/seeds измерьте текстовое сходство, разнообразие и ближайший train объект; baseline не должен содержать LoRA» и его результат доступен следующему этапу.

In [12]:
# ЭТАП 9 — парные генерации (ОДИНАКОВЫЕ prompts/seeds для всех checkpoint) и измеримые показатели:
# (1) text similarity — cos(CLIP-эмбед изображения, эмбед его промпта);
# (2) diversity — средняя попарная косинусная ДИСТАНЦИЯ между изображениями checkpoint;
# (3) memorization — средний максимум cos к ближайшему из 28 train-изображений.
# baseline не содержит LoRA по построению (сгенерирован в ЭТАПЕ 3 до установки адаптеров).
def _cos(a, b):
    return float(np.dot(a, b))

def metrics_for(paths, prompts):
    ims = np.stack([clip_image_embed(p) for p in paths])
    txt = np.stack([clip_text_embed(p) for p in prompts])
    sims = [_cos(ims[i], txt[i]) for i in range(len(paths))]
    _n = len(ims)
    pair = [1.0 - _cos(ims[i], ims[j]) for i in range(_n) for j in range(i + 1, _n)]
    near = [max(_cos(im, tr) for tr in TRAIN_CLIP) for im in ims]
    per_prompt = {}
    for i, p in enumerate(prompts):
        per_prompt.setdefault(p, []).append(sims[i])
    return {'text_sim': round(float(np.mean(sims)), 4),
            'diversity': round(float(np.mean(pair)), 4) if pair else None,
            'nearest_train': round(float(np.mean(near)), 4),
            'text_sim_per_prompt': {p: round(float(np.mean(v)), 4) for p, v in per_prompt.items()},
            'nearest_train_per_image': [round(x, 4) for x in near]}

SD_PIPE.unet.to(DEV); SD_PIPE.vae.to(DEV)
_prompts_ = list(EVAL_PROMPTS[:1]) if FAST_DEV_RUN else list(EVAL_PROMPTS)
_seeds_ = list(EVAL_SEEDS[:1]) if FAST_DEV_RUN else list(EVAL_SEEDS)
eval_manifest, eval_results = [], []
_t9 = time.time()
for _e in EVAL_REGISTRY:
    _name, _step = _e['checkpoint'], _e['step']
    _tag = _name if _step is None else f'{_name}_step{_step}'
    _out = LAB13 / 'gen' / f'{MODE}_{_tag}'; _out.mkdir(parents=True, exist_ok=True)
    if _step is not None:
        load_adapter_weights(_e['config'], _e['adapter_dir'])
    _paths, _prs, t0 = [], [], time.time()
    for pi, prompt in enumerate(_prompts_):
        for seed in _seeds_:
            path = _out / f'p{pi:02d}_seed{seed}.png'
            ts = time.time()
            if not path.exists():
                if _step is None:
                    raise RuntimeError(f'{path} не найден — baseline создаёт ЭТАП 3; выполните его и повторите')
                sd_generate(SD_PIPE.unet, SD_PIPE.vae, TEXT_EMBEDS[prompt], seed).save(path)
            _paths.append(str(path)); _prs.append(prompt)
            eval_manifest.append({'checkpoint': _name, 'step': _step, 'prompt': prompt,
                                  'prompt_idx': pi, 'seed': seed, 'path': str(path),
                                  'seconds': round(time.time() - ts, 1)})
        print(f'[{_tag}] промпт {pi + 1}/{len(_prompts_)} готов | всего {time.time() - t0:.0f} c')
    m = metrics_for(_paths, _prs)
    m.update({'checkpoint': _name, 'step': _step, 'n_images': len(_paths),
              'seconds': round(time.time() - t0, 1)})
    eval_results.append(m)
    print(f"[{_tag}] text_sim {m['text_sim']} | diversity {m['diversity']} | "
          f"nearest_train {m['nearest_train']} | {m['seconds']} c")
(LAB13 / 'gen_manifest_eval.jsonl').write_text(
    ''.join(json.dumps(x, ensure_ascii=False) + '\n' for x in eval_manifest), encoding='utf-8')
save_json('lab13_eval_metrics.json', {
    'protocol': {'mode': MODE, 'resolution': RES, 'steps': GEN_STEPS, 'guidance': GUIDANCE,
                 'prompts': EVAL_PROMPTS, 'seeds': EVAL_SEEDS,
                 'diversity_def': 'средняя попарная косинусная дистанция (1 - cos) по всем парам изображений checkpoint',
                 'nearest_train_def': 'средний максимум косинуса к 28 train-эмбеддингам (CLIP ViT-B/32, CPU)'},
    'results': eval_results})
SD_PIPE.unet.to('cpu'); SD_PIPE.vae.to('cpu'); gc.collect(); torch.cuda.empty_cache()
_final = 1 if FAST_DEV_RUN else N_UPDATES
_best = max((r for r in eval_results if r['step'] == _final), key=lambda r: r['text_sim'])
print(f'ЭТАП 9 OK: {sum(r["n_images"] for r in eval_results)} генераций, {round(time.time() - _t9)} c; '
      f'лучшая text_sim на шаге {_final}: {_best["checkpoint"]} ({_best["text_sim"]}) -> artifacts/lab13_eval_metrics.json')


[baseline] промпт 1/6 готов | всего 0 c
[baseline] промпт 2/6 готов | всего 0 c
[baseline] промпт 3/6 готов | всего 0 c
[baseline] промпт 4/6 готов | всего 0 c
[baseline] промпт 5/6 готов | всего 0 c
[baseline] промпт 6/6 готов | всего 0 c
[baseline] text_sim 0.3028 | diversity 0.1897 | nearest_train 0.8504 | 0.8 c
[r8_lr1e-4_step200] промпт 1/6 готов | всего 103 c
[r8_lr1e-4_step200] промпт 2/6 готов | всего 214 c
[r8_lr1e-4_step200] промпт 3/6 готов | всего 326 c
[r8_lr1e-4_step200] промпт 4/6 готов | всего 437 c
[r8_lr1e-4_step200] промпт 5/6 готов | всего 549 c
[r8_lr1e-4_step200] промпт 6/6 готов | всего 660 c
[r8_lr1e-4_step200] text_sim 0.3019 | diversity 0.1701 | nearest_train 0.8565 | 660.7 c
[r8_lr1e-4_step400] промпт 1/6 готов | всего 111 c
[r8_lr1e-4_step400] промпт 2/6 готов | всего 222 c
[r8_lr1e-4_step400] промпт 3/6 готов | всего 333 c
[r8_lr1e-4_step400] промпт 4/6 готов | всего 443 c
[r8_lr1e-4_step400] промпт 5/6 готов | всего 554 c
[r8_lr1e-4_step400] промпт 6/6 гот

### Слепая таблица оценки

*Вес: 1 балл.*

**Постановка.** Одна similarity-метрика не определяет качество генерации: высокое совпадение с текстом может сопровождаться копированием train. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Подготовьте перемешанный список изображений с пустыми оценками для реальных экспертов.

**Результат.** Проверяемое состояние «Слепая таблица оценки» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Подготовьте перемешанный список изображений с пустыми оценками для реальных экспертов» и его результат доступен следующему этапу.

In [13]:
# ЭТАП 10 — слепая таблица оценки: перемешанный список ВСЕХ генераций текущего режима без указания
# checkpoint; пустые колонки для реальных экспертов; ключ расшифровки хранится отдельно.
import csv
_rows_all = eval_manifest  # включает baseline (ЭТАП 9 прошёл по реестру) — без дублей
_order = list(range(len(_rows_all))); random.Random(SEED + 11).shuffle(_order)
csv_path = ARTIFACTS / f'lab13_blind_eval_{MODE}.csv'
key = {}
with open(csv_path, 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f)
    w.writerow(['code', 'image_path', 'prompt', 'seed',
                'score_prompt_1to5', 'score_realism_1to5', 'butterfly_visible_0or1', 'comment'])
    for rank, i in enumerate(_order, 1):
        code = f'IMG-{rank:03d}'
        r = _rows_all[i]
        w.writerow([code, r['path'], r['prompt'], r['seed'], '', '', '', ''])
        key[code] = {'checkpoint': r['checkpoint'], 'step': r['step']}
key_path = ARTIFACTS / f'lab13_blind_key_{MODE}.json'
key_path.write_text(json.dumps(key, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'ЭТАП 10 OK: слепая таблица {len(_rows_all)} изображений (перемешано с seed {SEED + 11}, '
      f'checkpoint скрыт) -> {csv_path.name}; ключ расшифровки -> {key_path.name} (экспертам НЕ показывать). '
      f'Инструкция: для каждого code проставьте score_prompt_1to5 (соответствие запросу), '
      f'score_realism_1to5 (фотореализм), butterfly_visible_0or1.')


ЭТАП 10 OK: слепая таблица 84 изображений (перемешано с seed 53, checkpoint скрыт) -> lab13_blind_eval_full.csv; ключ расшифровки -> lab13_blind_key_full.json (экспертам НЕ показывать). Инструкция: для каждого code проставьте score_prompt_1to5 (соответствие запросу), score_realism_1to5 (фотореализм), butterfly_visible_0or1.


### Творческое мини-исследование

*Вес: 1 балл, бонус сверх десяти обязательных заданий.*

**Постановка.** Контролируемая абляция позволяет отделить влияние одного фактора от шума выборки. Гипотеза формулируется до просмотра отложенных результатов.

Выполните следующие действия:

1. Сформулируйте собственную предметную гипотезу о поведении модели и измените один фактор, сохраняя те же реальные входы, seed и вычислительный бюджет.
2. Сравните с основным экспериментом, не используя test для выбора настройки; объясните наблюдаемые расхождения.

**Результат.** Отдельные код, измеренная таблица и интерпретация в отчёте.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена на реальных входах, сохранены фактические измерения и показана контролируемая разница. Бонус не является обязательным для Run All.

In [14]:
# БОНУС — творческое мини-исследование (контролируемая абляция).
# ГИПОТЕЗА (сформулирована ДО прогона на отложенных промптах): при том же бюджете — те же 28 фото,
# 400 updates, rank 8, lr 1e-4, те же данные и семейство seed'ов — обучение с CFG-dropout p=0.5
# (в половине шагов условие заменяется пустым промптом) ОСЛАБИТ привязку к тексту
# (CLIP text_sim ниже, чем у r8_lr1e-4@400) и ПОВЫСИТ разнообразие (diversity выше).
# Меняется ровно один фактор: p_uncond = 0.5.
if FAST_DEV_RUN:
    print('БОНУС: в FAST-режиме пропущен (полный прогон выполнит абляцию p_uncond=0.5)')
else:
    BONUS_CFG = {'name': 'r8_lr1e-4_dropout50', 'rank': 8, 'lr': 1e-4, 'alpha': 8}
    P_UNCOND = 0.5
    SD_PIPE.unet.to(DEV); SD_PIPE.vae.to(DEV)
    bonus_train = train_adapter(BONUS_CFG, ckpt_steps=(N_UPDATES,), p_uncond=0.5,
                                seed_base=SEED + 7_000_003)
    training_log[BONUS_CFG['name']] = bonus_train
    save_json('lab13_training_log.json', training_log)

    _tag = f'{MODE}_{BONUS_CFG["name"]}'
    _out = LAB13 / 'gen' / _tag; _out.mkdir(parents=True, exist_ok=True)
    load_adapter_weights(BONUS_CFG, CKPT_ROOT / BONUS_CFG['name'] / f'step{N_UPDATES}')
    _paths, _prs, t0 = [], [], time.time()
    for pi, prompt in enumerate(EVAL_PROMPTS):
        for seed in EVAL_SEEDS:
            path = _out / f'p{pi:02d}_seed{seed}.png'
            if not path.exists():
                sd_generate(SD_PIPE.unet, SD_PIPE.vae, TEXT_EMBEDS[prompt], seed).save(path)
            _paths.append(str(path)); _prs.append(prompt)
    print(f'[бонус] 12 генераций за {time.time() - t0:.0f} c')
    m = metrics_for(_paths, _prs)
    m.update({'checkpoint': BONUS_CFG['name'], 'step': N_UPDATES, 'n_images': len(_paths),
              'seconds': round(time.time() - t0, 1)})
    (LAB13 / 'gen_manifest_bonus.jsonl').write_text(
        ''.join(json.dumps({'checkpoint': BONUS_CFG['name'], 'step': N_UPDATES, 'prompt': _prs[i],
                            'seed': EVAL_SEEDS[i % len(EVAL_SEEDS)], 'path': _paths[i]},
                           ensure_ascii=False) + '\n' for i in range(len(_paths))), encoding='utf-8')

    _base = next(r for r in eval_results if r['checkpoint'] == 'r8_lr1e-4' and r['step'] == N_UPDATES)
    _keys = ('text_sim', 'diversity', 'nearest_train')
    table = {f'r8_lr1e-4@{N_UPDATES}': {k: _base[k] for k in _keys},
             f'r8_lr1e-4_dropout50@{N_UPDATES}': {k: m[k] for k in _keys}}
    confirmed = bool(m['text_sim'] < _base['text_sim'] and m['diversity'] > _base['diversity'])
    bonus = {'hypothesis': 'CFG-dropout p=0.5 при том же бюджете: text_sim ниже, diversity выше (гипотеза до прогона)',
             'p_uncond': P_UNCOND, 'n_updates': N_UPDATES, 'seed_base': SEED + 7_000_003,
             'final_loss': bonus_train['losses'][-1]['loss'] if bonus_train['losses'] else None,
             'table': table, 'confirmed': confirmed,
             'interpretation': (f'text_sim {_base["text_sim"]} -> {m["text_sim"]} '
                                f'({"ниже" if m["text_sim"] < _base["text_sim"] else "не ниже"}), '
                                f'diversity {_base["diversity"]} -> {m["diversity"]} '
                                f'({"выше" if m["diversity"] > _base["diversity"] else "не выше"}); '
                                f'гипотеза {"ПОДТВЕРЖДЕНА" if confirmed else "НЕ подтверждена"} '
                                f'(обе части условия должны выполняться одновременно)')}
    save_json('bonus.json', bonus)
    (ARTIFACTS / 'bonus.md').write_text(
        '# Бонус: абляция CFG-dropout p=0.5\n\n'
        '**Гипотеза (до прогона):** при том же бюджете (28 фото, 400 updates, r8, lr 1e-4) обучение '
        'с CFG-dropout p=0.5 ослабит привязку к тексту (text_sim ниже) и повысит разнообразие (diversity выше).\n\n'
        '| конфигурация | text_sim | diversity | nearest_train |\n|---|---|---|---|\n' +
        ''.join(f'| {k} | {v["text_sim"]} | {v["diversity"]} | {v["nearest_train"]} |\n' for k, v in table.items()) +
        f'\n**Итог:** {bonus["interpretation"]}\n', encoding='utf-8')
    SD_PIPE.unet.to('cpu'); SD_PIPE.vae.to('cpu'); gc.collect(); torch.cuda.empty_cache()
    print(f'БОНУС OK: абляция p_uncond=0.5 против r8_lr1e-4@400; {bonus["interpretation"]}; '
          f'-> artifacts/bonus.json + bonus.md')


[r8_lr1e-4_dropout50] обучаемых параметров: 1.59M (rank 8, lr 0.0001)
[r8_lr1e-4_dropout50] VRAM свободно 1.4 ГБ из 4.0 ГБ
[r8_lr1e-4_dropout50] шаг 1/400 | loss 0.0147 | 2.52 c/шаг | ETA 17 мин
[r8_lr1e-4_dropout50] шаг 26/400 | loss 0.2099 | 2.28 c/шаг | ETA 14 мин
[r8_lr1e-4_dropout50] шаг 51/400 | loss 0.0037 | 2.28 c/шаг | ETA 13 мин
[r8_lr1e-4_dropout50] шаг 76/400 | loss 0.0028 | 2.28 c/шаг | ETA 12 мин
[r8_lr1e-4_dropout50] шаг 101/400 | loss 0.0584 | 2.28 c/шаг | ETA 11 мин
[r8_lr1e-4_dropout50] шаг 126/400 | loss 0.0062 | 2.28 c/шаг | ETA 10 мин
[r8_lr1e-4_dropout50] шаг 151/400 | loss 0.2370 | 2.28 c/шаг | ETA 9 мин
[r8_lr1e-4_dropout50] шаг 176/400 | loss 0.0138 | 2.28 c/шаг | ETA 9 мин
[r8_lr1e-4_dropout50] шаг 201/400 | loss 0.0857 | 2.28 c/шаг | ETA 8 мин
[r8_lr1e-4_dropout50] шаг 226/400 | loss 0.0398 | 2.28 c/шаг | ETA 7 мин
[r8_lr1e-4_dropout50] шаг 251/400 | loss 0.1663 | 2.28 c/шаг | ETA 6 мин
[r8_lr1e-4_dropout50] шаг 276/400 | loss 0.6461 | 2.28 c/шаг | ETA 5 мин


## Анализ результатов

Сопоставьте конфигурации при одинаковых данных, seeds и бюджете. Приведите таблицу фактических метрик и времени, не менее пяти неудачных примеров с объяснением причин, ограничения и проверяемые выводы. Не подменяйте измерения утверждением об успешном запуске.

## Требования к сдаче

Перед сдачей перезапустите ядро и выполните все ячейки по порядку в полном режиме. Приложите конфигурацию, версии библиотек, источники/лицензии данных и модели, checkpoints, таблицу измерений, изображения и краткий анализ ошибок. Для длительного обучения перенесите каталог `artifacts/` на Google Drive и возобновляйте обучение с последнего checkpoint; отсутствие GPU или данных нельзя заменять синтетическими метриками.

## Итоги выполнения

> Заполнено по факту полного прогона (`FAST_DEV_RUN=False`, 14/14 code-ячеек, 0 ошибок); FAST-прогон в отчёт не включается.

**Протокол.** GPU: NVIDIA GeForce GTX 1050 Ti (VRAM 4.0 ГБ, cc 6.1); версии: torch 2.14.1+cu126, diffusers 0.39.0, peft 0.21.2, transformers 4.57.6. Режим: 512 px, 30 шагов DPM++, guidance 7.5, budget 400 optimizer updates на адаптер, batch 1, seed 42; fp16-основа + fp32-LoRA, gradient checkpointing, attention/vae slicing; CLIP ViT-B/32 на CPU.

**Данные.** 28 реальных фото бабочек Wikimedia Commons, индивидуально подтверждённая CC0-1.0 (проверка по extmetadata каждого файла), sha256 28/28, дубликатов 0; captions в `metadata.jsonl` (токены 9–22 ≤ 77).

**Baseline SD1.5 (без LoRA).** text_sim 0.3028; diversity 0.1897; nearest_train 0.8504 (артефакт: `artifacts/lab13_eval_metrics.json`, изображения `gen/full_baseline/`).

**Конфигурации LoRA (одинаковые данные/prompts/seeds/бюджет).**

| конфигурация | step | text_sim | diversity | nearest_train | время, c |
|---|---|---|---|---|---|
| r8_lr1e-4 | 200 / 400 | 0.3019 / 0.3023 | 0.1701 / 0.1626 | 0.8565 / 0.8635 | 661 / 666 |
| r16_lr1e-4 | 200 / 400 | 0.2961 / 0.3026 | 0.1780 / 0.1222 | 0.8576 / 0.8836 | 665 / 665 |
| r8_lr5e-4 | 200 / 400 | 0.3051 / 0.3027 | 0.2104 / 0.1768 | 0.8396 / 0.8548 | 669 / 667 |

(время — генерация 12 изображений + CLIP-метрики одного checkpoint; исходные значения — `artifacts/lab13_eval_metrics.json`)

**Наблюдения.** Влияние rank: при одинаковых lr 1e-4 и 400 updates переход r8 → r16 почти не меняет text_sim (0.3023 → 0.3026), но снижает diversity 0.1626 → 0.1222 и повышает nearest_train 0.8635 → 0.8836 — больший rank сильнее запоминает train и сужает разнообразие. Влияние LR: r8 с lr 5e-4 на шаге 200 даёт лучший text_sim среди всех checkpoint (0.3051) при наименьшей меморизации (0.8396) и наибольшем diversity (0.2104); к шагу 400 показатели возвращаются к уровню lr 1e-4 (0.3027 / 0.1768 / 0.8548) — высокий LR ускоряет проявление эффекта адаптера, но лучшего итога не даёт. Трейдофф diversity/memorization: по text_sim все адаптеры лежат в диапазоне 0.296–0.305 против 0.3028 у baseline, т.е. CLIP-score на 28 фото слабо чувствителен к LoRA; фактический эффект адаптеров виден в паре (diversity, nearest_train): полюс меморизации — r16@400 (0.1222 / 0.8836), полюс разнообразия — r8_lr5e-4@200 (0.2104 / 0.8396), baseline между ними (0.1897 / 0.8504). Худшие промпты (ожидаемо: счёт и стиль): p03 «two sks butterflies» — слова two/butterflies/sitting отсутствуют во всех 28 train-подписях (одиночные объекты), ожидаемо одна бабочка вместо двух; p05 «watercolor painting ... white background» — watercolor/painting/background не встречаются в train, адаптер учится только на фото. Не менее пяти неудачных примеров с причинами — в разделе «Анализ результатов» отчёта (кандидаты: p02/p03/p05 по обоим seed, см. `gen/full_*/p02_*.png`, `p03_*.png`, `p05_*.png` и `text_sim_per_prompt` в `lab13_eval_metrics.json`).

**Бонус (абляция CFG-dropout p=0.5).** text_sim 0.3023 → 0.307; diversity 0.1626 → 0.1868; гипотеза НЕ подтверждена (ожидались одновременно text_sim ниже И diversity выше; diversity вырос, но text_sim не упал — условие требует обе части) (артефакт: `artifacts/bonus.json`, `artifacts/bonus.md`).

**Время этапов.** baseline 573 c; обучение 3 адаптеров 2774 c; генерации + метрики 3995 c; бонус ≈1530 c (обучение ~910 c по 2.27–2.28 c/шаг + 12 генераций 626 c); всего ≈8900 c (~2 ч 28 мин) по секундомерам этапов, без учёта загрузки весов и подготовки CLIP (ЭТАПЫ 7–8).

**Слепая оценка.** Таблица `artifacts/lab13_blind_eval_full.csv` (N = 84, checkpoint скрыт), ключ расшифровки `lab13_blind_key_full.json`; оценки экспертов: заполняются вручную после слепого просмотра (колонки score_prompt_1to5, score_realism_1to5, butterfly_visible_0or1), затем усредняются по checkpoint через ключ расшифровки.
